In [39]:

import pandas as pd
import os
from openai import OpenAI
import plotly.express as px
from IPython.display import display, HTML
from dotenv import load_dotenv
load_dotenv()

True

In [27]:

client = OpenAI(
	api_key=os.getenv("OPENAI_API_KEY"),
)

In [28]:
def load_data(file_path):
    return pd.read_csv(file_path)

In [29]:
def preprocess_data(df):
    # Remove any empty rows or columns
    df = df.dropna(how='all').dropna(axis=1, how='all')
    return df

In [49]:
def generate_summary(data):
    prompt = f"""
    *Prompt 1: Section Brief Summary of Adverse Events*
    You will be given tables in the variable 'input tables' which includes information about brief summary of adverse events that occurred during a clinical trial. Identify the data according to the instructions. You will be given the necessary context to answer certain questions. Avoid fabricating responses. Do not elaborate.
    Context: Given the following data from a clinical trial, summarize the key information about the total number of subjects with Treatment Emergent Adverse Event(TEAE) and Serious Adverse Events(SAE).
    Below are multiple table details. For each table you are given the table title, i.e., "table title: " which tells what information is in the table, and the table itself, i.e. "table: ". With the table title and table content, follow the below task.
    Task: Summarize the table title and then provide the table reference (you will find in this format x.x.x.x. where x is integer). An example of how to begin the summary: A summary of treatment related adverse events (TEAE) is provided in table 14.2.1
    Summarize the results for each cohort of the population in the clinical trial in separate paragraphs. Cohorts may include treatment arms, phases, parts, cohorts, periods and other categories (e.g., in the drug xx treatment arm 3 subjects experienced a treatment-emergent adverse event). The summary should be about AEs or SARs or TEAEs.

    Input tables:

    {data.to_string()}

    Please provide a summary based on the above instructions and data.
    """

    response = client.chat.completions.create(
        model="gpt-4o-mini",  # or another suitable model
        messages=[
            {"role": "system", "content": "You are a clinical trial data analyst specializing in summarizing adverse event data."},
            {"role": "user", "content": prompt}
        ],
        max_tokens=1000
    )

    return response.choices[0].message.content

In [32]:

file_path = 'Brief Summary of Adverse Events.csv'
df = load_data(file_path)
processed_df = preprocess_data(df)

In [43]:
print("Processed Data:")
display(processed_df)

Processed Data:


,Unnamed: 0,Vehicle,0.10%,0.50%,1.00%,5.00%,Vehicle.1,0.50%.1,1.00%.1,Total
0,NaN,QD,QD,QD,QD,QD,TID,TID,TID,NaN
1,Number (%) of subjects,n (%),n (%),n (%),n (%),n (%),n (%),n (%),n (%),n (%)
2,Subjects evaluable for adverse events,39,39,36,39,36,36,36,39,300
3,Number of adverse events,28,24,19,23,13,27,13,19,166
4,Subjects with adverse events,18 (46.2),17 (43.6),11 (30.6),12 (30.8),10 (27.8),17 (47.2),9 (25.0),14 (35.9),108 (36.0)
5,Subjects with severe adverse events,0,0,0,0,0,1 (2.8),1 (2.8),0,2 (0.7)
6,Subject discontinued from study due to adverse...,3 (7.7),2 (5.1),1 (2.8),2 (5.1),2 (5.6),5 (13.9),1 (2.8),0,16 (3.9)
7,Subjects discontinued study drug due to AE and...,0,2 (5.1),0,0,0,1 (2.8),0,0,3 (1.0)
8,Subjects with temporary discontinuation due to...,3 (7.7),1 (2.6),0,0,1 (2.8),0,0,0,5 (1.8)


In [50]:
summary = generate_summary(processed_df)
print("\nGenerated Summary:")
print(summary)


Generated Summary:
A summary of treatment emergent adverse events (TEAE) is provided in table 14.2.1. 

In the Vehicle cohort, among 39 evaluable subjects, 18 (46.2%) experienced adverse events, with no severe adverse events reported. Additionally, 3 subjects discontinued the study due to adverse events. 

In the 0.10% treatment arm, 39 subjects were evaluable, and 17 (43.6%) had adverse events, also with no severe adverse events noted. Two subjects discontinued the study due to adverse events. 

For the 0.50% treatment arm, 36 evaluable subjects were reported, with 11 (30.6%) experiencing adverse events, and again, no severe adverse events occurred. One subject discontinued the study due to an adverse event. 

In the 1.00% treatment group, among 39 evaluable subjects, 12 (30.8%) had adverse events, with no severe adverse events noted. Two subjects also discontinued the study due to adverse events. 

For the 5.00% treatment arm, 36 subjects were evaluable, where 10 (27.8%) reported ad

In [37]:

summary = generate_summary(processed_df)
print("\nGenerated Summary:")
print(summary)


Generated Summary:
A summary of treatment-emergent adverse events (TEAE) and serious adverse events (SAE) during the clinical trial is captured in the input tables. A total number of 300 subjects were evaluated for adverse events. 

Out of these patients, a whole sum of 166 adverse events was reported. When further categorized by the treatment sub-groups, it was noted that the frequency of adverse events varied. The maximum number of adverse events 28 (46.2%) was seen in patients who received Vehicle treatment QD, followed by 27 (47.2%) who received Vehicle treatment TID. At the same token, it was recorded that minimum adverse events were noted in the group treated with 0.50% TID, with 13 incidents reported.

Despite the varying number of adverse events, serious adverse events were relatively low, with only 2 (0.7%) reported. One severe adverse event was reported from patients who received two different treatment types, namely, Vehicle TID and 0.50% TID. 

Regarding treatment disconti

In [47]:

display(HTML(summary.replace('\n', '<br>')))